# Практическая работа 3
Распределённая модель обработки данных

Выполнены учебный анализ Olist и самостоятельный анализ продаж КОМУС. Источник Olist восстановлен из CSV второй практики; фактическая версия содержит 100 000 отзывов вместо 97 621 в примере методички. Числа вычисляются, а не подставляются из задания.

Студент: Албахтин И.В., ИНБО-12-23. Отдельная VM TIABD-Albakhtin-IV.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark import StorageLevel
from lab_helpers import environment, records, save_json, plan, measured, checkpoint, ui_data, schema_signature
ROOT = Path.cwd()
STAGING = ROOT / "data" / "spark_input"
OUT = ROOT / "exports"
OUT.mkdir(exist_ok=True)
def read_csv(name, schema=None, infer=False):
    reader = spark.read.option("header", True).option("encoding", "UTF-8")
    if schema is not None: reader = reader.schema(schema)
    return reader.option("inferSchema", infer).csv(str(STAGING / f"{name}.csv"))

spark = (SparkSession.builder.master("local[*]").appName("TIABD_Practice_3_Albakhtin_IV")
    .config("spark.driver.memory", "1536m").config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.bindAddress", "0.0.0.0").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
result = {"environment": environment(spark), "source": json.loads((ROOT/"source_manifest.json").read_text())}
print(json.dumps(result["environment"], ensure_ascii=False, indent=2))


26/10/05 19:41:13 WARN Utils: Your hostname, tiabd-albakhtin-iv resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/05 19:41:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/05 19:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{
  "python": "3.10.12",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "spark": "3.5.7",
  "os": "Linux-5.15.0-198-generic-x86_64-with-glibc2.35",
  "cpu": "Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           46 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  2\nOn-line CPU(s) list:                     0,1\nVendor ID:                               AuthenticAMD\nModel name:                              AMD Ryzen 7 9700X 8-Core Processor\nCPU family:                              26\nModel:                                   68\nThread(s) per core:                      1\nCore(s) per socket:                      2\nSocket(s):                               1\nStepping:                                0\nBogoMIPS:                                7599.98\nFlags:                                   fpu vme de pse t

## 1 SQLite и staging
Стандартный sqlite3 открывает источник. CSV отделяет выгрузку из СУБД от аналитики Spark; все 11 таблиц доступны для проверки.

In [2]:
import sqlite3
import pandas as pd
names = ["orders", "order_items", "products", "customers", "order_payments", "order_reviews", "product_category_name_translation"]
with sqlite3.connect(ROOT/"data"/"ecommerce.sqlite") as con:
    counts = {}
    for name in names:
        total = 0
        for i, chunk in enumerate(pd.read_sql_query(f'SELECT * FROM "{name}"', con, chunksize=50000)):
            chunk.to_csv(STAGING/f"{name}.csv", index=False, header=i==0, mode="w" if i==0 else "a")
            total += len(chunk)
        counts[name] = total
        print(name, total)
result["source_counts"] = counts


orders 99441


order_items 112650
products 32951


customers 99441
order_payments 103886


order_reviews 100000
product_category_name_translation 71


## 2 Схемы и типы
При обычном чтении CSV все значения строковые. StructType фиксирует числовые типы позиций, а временные поля заказов преобразуются в timestamp.

In [3]:
orders_raw = read_csv("orders")
orders_raw.printSchema()
orders_raw.select("order_status", "order_purchase_timestamp").show(5, False)
items_schema = T.StructType([T.StructField(c, typ, True) for c, typ in [
    ("order_id", T.StringType()), ("order_item_id", T.IntegerType()),
    ("product_id", T.StringType()), ("seller_id", T.StringType()),
    ("shipping_limit_date", T.StringType()), ("price", T.DoubleType()), ("freight_value", T.DoubleType())]])
items = read_csv("order_items", items_schema)
items.printSchema()
items_infer = read_csv("order_items", infer=True)
print("Explicit equals inferred:", items.schema == items_infer.schema)
timestamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = orders_raw
for c in timestamp_cols: orders = orders.withColumn(c, F.to_timestamp(c, "yyyy-MM-dd HH:mm:ss"))
orders.printSchema()
result["schemas"] = {"raw": orders_raw.schema.jsonValue(), "items": items.schema.jsonValue(), "orders": orders.schema.jsonValue()}


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



+------------+------------------------+
|order_status|order_purchase_timestamp|
+------------+------------------------+
|delivered   |2017-10-02 10:56:33     |
|delivered   |2018-07-24 20:41:37     |
|delivered   |2018-08-08 08:38:49     |
|delivered   |2017-11-18 19:28:06     |
|delivered   |2018-02-13 21:18:39     |
+------------+------------------------+
only showing top 5 rows

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



Explicit equals inferred: False
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



## 3 Преобразования и пропуски
У дорогих позиций рассчитывается стоимость с доставкой. Отсутствующая дата не заменяется фиктивной: статус заказа объясняет её бизнес-смысл.

In [4]:
expensive = (items.filter(F.col("price") >= 500).withColumn("gross_item_value", F.round(F.col("price")+F.col("freight_value"), 2))
    .select("order_id", "order_item_id", "price", "freight_value", "gross_item_value"))
result["expensive_count"] = expensive.count()
result["expensive_top"] = records(expensive.orderBy(F.desc("gross_item_value")).limit(5))
expensive.drop("order_id").orderBy(F.desc("gross_item_value")).show(5, False)
print("Price >= 500:", result["expensive_count"])
delivered = (orders.filter(F.col("order_status")=="delivered")
    .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("is_late", F.col("order_delivered_customer_date")>F.col("order_estimated_delivery_date")))
known = delivered.filter(F.col("order_delivered_customer_date").isNotNull())
result["delivery"] = {"delivered": delivered.count(), "known": known.count(), "late": known.filter("is_late").count()}
print(result["delivery"])
result["nulls"] = orders.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in timestamp_cols]).first().asDict()
print(result["nulls"])
missing = orders.filter(F.col("order_delivered_customer_date").isNull()).groupBy("order_status").count().orderBy(F.desc("count"))
missing.show(False) if False else missing.show(truncate=False)
result["missing_status"] = records(missing)
products = read_csv("products", infer=True)
result["missing_categories"] = products.filter(F.col("product_category_name").isNull()).count()
products_clean = products.fillna({"product_category_name": "unknown"})
print("Products without category:", result["missing_categories"])


+-------------+------+-------------+----------------+
|order_item_id|price |freight_value|gross_item_value|
+-------------+------+-------------+----------------+
|1            |6735.0|194.31       |6929.31         |
|1            |6729.0|193.21       |6922.21         |
|1            |6499.0|227.66       |6726.66         |
|1            |4799.0|151.34       |4950.34         |
|1            |4690.0|74.34        |4764.34         |
+-------------+------+-------------+----------------+
only showing top 5 rows

Price >= 500: 3239


{'delivered': 96478, 'known': 96470, 'late': 7826}


{'order_purchase_timestamp': 0, 'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965, 'order_estimated_delivery_date': 0}


+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



Products without category: 610


## 4 Агрегации и JOIN
Платёжные строки не тождественны заказам. LEFT JOIN сохраняет факты и позволяет измерить неполноту справочника.

In [5]:
status = orders.groupBy("order_status").count().orderBy(F.desc("count"))
status.show(truncate=False)
payments = read_csv("order_payments", infer=True)
payment_stats = payments.groupBy("payment_type").agg(F.count("*").alias("payments"), F.round(F.sum("payment_value"),2).alias("total_value"), F.round(F.avg("payment_value"),2).alias("avg_value")).orderBy(F.desc("total_value"))
payment_stats.show(truncate=False)
translations = read_csv("product_category_name_translation")
sales = (items.alias("i").join(products_clean.select("product_id","product_category_name").alias("p"), "product_id", "left")
    .join(translations.alias("t"), "product_category_name", "left")
    .select("i.*", F.coalesce("product_category_name_english", "product_category_name", F.lit("unknown")).alias("category")))
result["join"] = {"before": items.count(), "after": sales.count(), "unknown": sales.filter("category='unknown'").count()}
print("JOIN checks:", result["join"])
customers = read_csv("customers", infer=True)
late_state = (known.join(customers.select("customer_id","customer_state"), "customer_id", "left").groupBy("customer_state")
    .agg(F.count("*").alias("delivered"),F.sum(F.col("is_late").cast("int")).alias("late"))
    .withColumn("late_pct", F.round(F.col("late")/F.col("delivered")*100,2)).orderBy(F.desc("late_pct")))
late_state.show(10, False)
result.update(status=records(status), payments=records(payment_stats), late_state=records(late_state.limit(10)))


+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



JOIN checks: {'before': 112650, 'after': 112650, 'unknown': 1603}


+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows



## 5 Spark SQL и экспорт
SQL и DataFrame API формируют планы одного движка. Проверяется равенство всех групп, а не только первых строк.

In [6]:
sales.createOrReplaceTempView("sales")
category_sql = spark.sql("""SELECT category, COUNT(*) items, COUNT(DISTINCT order_id) orders,
    ROUND(SUM(price),2) revenue, ROUND(AVG(price),2) avg_price FROM sales GROUP BY category""")
category_api = sales.groupBy("category").agg(F.count("*").alias("items"), F.countDistinct("order_id").alias("orders"),F.round(F.sum("price"),2).alias("revenue"),F.round(F.avg("price"),2).alias("avg_price"))
diff = category_sql.exceptAll(category_api).count()+category_api.exceptAll(category_sql).count()
assert diff == 0
category_api.orderBy(F.desc("revenue")).show(10, False)
category_api.write.mode("overwrite").parquet(str(OUT/"pr3_olist_parquet"))
category_api.coalesce(1).write.mode("overwrite").option("header",True).csv(str(OUT/"pr3_olist_csv"))
back = spark.read.parquet(str(OUT/"pr3_olist_parquet"))
assert back.count()==category_api.count() and schema_signature(back)==schema_signature(category_api)
result["category"] = records(category_api.orderBy(F.desc("revenue")))
result["export"] = {"rows": back.count(), "types_equal": schema_signature(back)==schema_signature(category_api), "nullable_equal": back.schema==category_api.schema, "sql_diff": diff}
print(result["export"])


+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows



{'rows': 74, 'types_equal': True, 'nullable_equal': False, 'sql_diff': 0}


## 6 Самостоятельная работа КОМУС
865 222 позиции продаж и отдельный справочник товаров. Восемь полей факта, четыре числовых и несколько категориальных признаков. Из входа исключены адреса, клиентские реквизиты и номера автомобилей. Условия публичного распространения исходного архива не предоставлены; исходные строки остаются локальными.

In [7]:
own_path = ROOT/"data"/"komus"
own_schema = T.StructType([T.StructField(c,T.StringType(),True) for c in ["cost","purchase_date","receipt_id","product_key","branch","quantity","amount","discount"]])
raw = spark.read.option("header",True).schema(own_schema).csv(str(own_path/"sales.csv"))
inferred = spark.read.option("header",True).option("inferSchema",True).csv(str(own_path/"sales.csv"))
raw.printSchema()
print("Rows:", raw.count(), "Columns:", raw.columns)
inferred.printSchema()
fact = raw.withColumn("purchase_date", F.to_date("purchase_date","dd.MM.yyyy"))
for c in ["cost","quantity","amount","discount"]:
    fact = fact.withColumn(c, F.regexp_replace(c, ",", ".").cast("double"))
fact = (fact.withColumn("branch",F.trim("branch")).withColumn("month",F.date_format("purchase_date","yyyy-MM"))
    .withColumn("margin", F.col("amount")-F.col("cost")))
own_nulls = fact.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["receipt_id","purchase_date","amount","cost","product_key"]]).first().asDict()
print("Nulls:", own_nulls)
good = fact.filter(F.col("purchase_date").isNotNull() & F.col("amount").isNotNull())
# Missing cost prevents margin calculation, not sales analysis. Keep the sale
# and measure cost coverage explicitly rather than fabricate a zero cost.
# Receipt IDs may be structurally absent: keep those sales, never fill with a real ID.
print("Rows retained:", good.count())
dim_raw = spark.read.option("header",True).csv(str(own_path/"products.csv"))
dim = dim_raw.select("product_key","brand","product_group").dropDuplicates()
conflicts = dim.groupBy("product_key").count().filter("count>1").count()
assert conflicts == 0, 'Ambiguous product dimension'
dim = dim.fillna({"brand":"unknown","product_group":"unknown"})
left = good.join(dim,"product_key","left").fillna({"brand":"unknown","product_group":"unknown"})
inner = good.join(dim,"product_key","inner")
join_counts = {"before":good.count(),"left":left.count(),"inner":inner.count(),"unmatched":good.join(dim,"product_key","left_anti").count()}
print("JOIN:",join_counts)
left.printSchema()
result["komus"] = {"rows": raw.count(), "columns": raw.columns, "nulls":own_nulls,"valid":good.count(),"dim_raw":dim_raw.count(),"dim_unique":dim.count(),"conflicts":conflicts,"joins":join_counts,"schema":left.schema.jsonValue()}


root
 |-- cost: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- receipt_id: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- discount: string (nullable = true)

Rows: 865222 Columns: ['cost', 'purchase_date', 'receipt_id', 'product_key', 'branch', 'quantity', 'amount', 'discount']
root
 |-- cost: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- receipt_id: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- discount: string (nullable = true)



Nulls: {'receipt_id': 3934, 'purchase_date': 0, 'amount': 0, 'cost': 86431, 'product_key': 0}


Rows retained: 865222


JOIN: {'before': 865222, 'left': 865222, 'inner': 865222, 'unmatched': 0}
root
 |-- product_key: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- receipt_id: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- month: string (nullable = true)
 |-- margin: double (nullable = true)
 |-- brand: string (nullable = false)
 |-- product_group: string (nullable = false)



## 7 Группировки КОМУС и SQL
Маржа определяется как сумма покупки минус предоставленная себестоимость строки. Это расчётный показатель, а не чистая прибыль: налоги и дополнительные расходы в данных отсутствуют.

In [8]:
group_stats = left.groupBy("product_group").agg(F.count("*").alias("lines"), F.count("cost").alias("cost_known"), F.round(F.sum("amount"),2).alias("revenue"), F.round(F.sum("margin"),2).alias("margin"),F.round(F.avg("amount"),2).alias("avg_line"))
branch_stats = left.groupBy("branch").agg(F.count("*").alias("lines"),F.round(F.sum("amount"),2).alias("revenue"),F.round(F.sum("quantity"),2).alias("units")).orderBy(F.desc("revenue"))
group_stats.orderBy(F.desc("revenue")).show(10, False)
branch_stats.show(10, False)
left.createOrReplaceTempView("komus_sales")
sql1 = spark.sql("""SELECT product_group, COUNT(*) lines, COUNT(cost) cost_known, ROUND(SUM(amount),2) revenue,
    ROUND(SUM(margin),2) margin, ROUND(AVG(amount),2) avg_line
    FROM komus_sales GROUP BY product_group ORDER BY revenue DESC""")
sql2 = spark.sql("""SELECT month, COUNT(*) lines, ROUND(SUM(amount),2) revenue,
    ROUND(SUM(discount),2) discount FROM komus_sales GROUP BY month ORDER BY month""")
sql2.show(30,False)
assert group_stats.exceptAll(sql1).count()+sql1.exceptAll(group_stats).count()==0
group_stats.write.mode("overwrite").parquet(str(OUT/"pr3_komus_parquet"))
group_stats.coalesce(1).write.mode("overwrite").option("header",True).csv(str(OUT/"pr3_komus_csv"))
own_back = spark.read.parquet(str(OUT/"pr3_komus_parquet"))
assert own_back.count()==group_stats.count() and schema_signature(own_back)==schema_signature(group_stats)
result["komus"].update(groups=records(group_stats.orderBy(F.desc("revenue"))),branches=records(branch_stats),monthly=records(sql2),output_rows=own_back.count(),roundtrip_types_equal=schema_signature(own_back)==schema_signature(group_stats),roundtrip_nullable_equal=own_back.schema==group_stats.schema)
save_json("pr3.json",result)
checkpoint("pr3_completed",spark)


+----------------------------------+-----+----------+-------------+----------+--------+
|product_group                     |lines|cost_known|revenue      |margin    |avg_line|
+----------------------------------+-----+----------+-------------+----------+--------+
|ЗХК                               |77998|70271     |1.101367173E7|4970760.76|141.2   |
|Основы-холст на подрамнике-Базовый|6703 |6010      |4407000.88   |2022021.65|657.47  |
|Бумага-Картон-пенокартон          |2531 |2295      |2716844.79   |1213973.44|1073.43 |
|Faber-Castell                     |10023|8949      |2632258.18   |1175326.88|262.62  |
|Подрамники, рамы-Планшеты         |3850 |3466      |2550475.59   |1170236.14|662.46  |
|Сумки-на молнии ткань/кожа        |2949 |2640      |2184469.42   |983199.08 |740.75  |
|Кисти-Масло-Синтетика             |24405|21907     |2102178.52   |941226.76 |86.14   |
|Гамма                             |19163|17331     |1889976.3    |857073.08 |98.63   |
|Мольберты-Этюдники             

+----------------------------+------+-------------+--------+
|branch                      |lines |revenue      |units   |
+----------------------------+------+-------------+--------+
|ТЗ-2 ( Ленинградское шоссе) |431372|5.875080324E7|661375.0|
|ТЗ-1 ( ул.Фадеева)          |238046|4.434316832E7|472509.0|
|ТЗ-3 (Курский)              |190730|2.624905561E7|285798.0|
|Багетная мастерская         |848   |1399096.07   |993.0   |
|РС (Распределительный склад)|249   |788679.75    |11500.0 |
|Кафе-1 ( ул.Фадеева)        |3977  |207762.35    |4990.0  |
+----------------------------+------+-------------+--------+



+-------+------+-------------+---------+
|month  |lines |revenue      |discount |
+-------+------+-------------+---------+
|2020-06|2932  |269717.58    |57212.82 |
|2020-07|13218 |1317945.84   |208144.55|
|2020-08|11918 |1309760.8    |15661.86 |
|2020-09|20421 |2121197.1    |-0.18    |
|2020-10|21995 |2684682.71   |20035.98 |
|2020-11|20728 |2245580.97   |1824.02  |
|2020-12|25501 |3386536.22   |62.57    |
|2021-01|20506 |2562761.24   |7760.76  |
|2021-02|25085 |2972448.33   |6.84     |
|2021-03|24446 |2829779.86   |146.7    |
|2021-04|21864 |2542105.5    |162.5    |
|2021-05|29985 |3534211.82   |167.2    |
|2021-06|32742 |4308661.66   |14207.21 |
|2021-07|61749 |1.11930306E7 |130975.48|
|2021-08|69165 |1.163173015E7|210064.63|
|2021-09|120121|1.90532626E7 |434489.79|
|2021-10|107459|1.744767883E7|314085.35|
|2021-11|108614|1.730314501E7|351826.41|
|2021-12|126773|2.302432852E7|509723.89|
+-------+------+-------------+---------+



SCREENSHOT CHECKPOINT pr3_completed


## 8 Выводы
1. CSV требует контроля типов; десятичная запятая КОМУС не распознаётся как число автоматически.
2. Пропуски дат доставки Olist зависят от статуса.
3. LEFT JOIN сохраняет неполные факты; INNER JOIN позволяет проверить охват справочника.
4. SQL и DataFrame API дали одинаковые агрегаты.
5. Parquet сохранил схему и количество групп.

Ограничения: local[*] не является физическим кластером; значения маржи не учитывают все расходы предприятия, а версия Olist отличается от примера методички.

In [9]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
